In [1]:
import sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Adjust import path if data_utils.py is placed at repo root instead of pkg/
from pkg.data_utils import preprocess_data
from pkg.eval import regression_metrics

In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

X_train, X_test, y_train, y_test = preprocess_data()

results = []

# Baseline 1: Constant mean of y_train
mean_pred = np.full(shape=len(y_test), fill_value=y_train.mean())
results.append({"Model": "Baseline (Train Mean)", **regression_metrics(y_test, mean_pred)})

# Baseline 2: Carry forward scaled Pre_Semester_GPA
if "Pre_Semester_GPA" in X_test.columns:
    gpa_anchor_pred = X_test["Pre_Semester_GPA"].to_numpy()
    results.append(
        {"Model": "Baseline (Pre_Semester_GPA)", **regression_metrics(y_test, gpa_anchor_pred)}
    )

# XGBoost on preprocessed features
xgb_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

results.append({"Model": "XGBRegressor", **regression_metrics(y_test, xgb_pred)})

metrics_df = pd.DataFrame(results)[["Model", "MSE", "RMSE", "MAE", "R2"]]
print(metrics_df.to_string(index=False))

                      Model       MSE     RMSE      MAE         R2
      Baseline (Train Mean)  0.241423 0.491348 0.403681  -0.000156
Baseline (Pre_Semester_GPA) 11.530621 3.395677 3.347900 -46.768450
               XGBRegressor  0.020490 0.143144 0.112682   0.915114
